# HomeWizard Plug-In Battery: opbrengst op eigen data
Uursimulatie (ontladen alleen zonder teruglevering) voor 1 en 2 modules, 2,47 kWh bruikbaar en 800 W per module.

In [ ]:
import sqlite3, os, time
import numpy as np, pandas as pd

RENDEMENT = 0.75
GEEN_TERUG = 0.01
DAGEN = 400
DB = next(p for p in ["/config/home-assistant_v2.db", "/homeassistant/home-assistant_v2.db"] if os.path.exists(p))
IMPORT_IDS = ["sensor.slimmemeter_energy_delivered_tariff_1", "sensor.slimmemeter_energy_delivered_tariff_2"]
EXPORT_IDS = ["sensor.slimmemeter_energy_returned_tariff_1", "sensor.slimmemeter_energy_returned_tariff_2"]
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)

def reeks(sid):
    mid, unit = con.execute("SELECT id, unit_of_measurement FROM statistics_meta WHERE statistic_id=?", (sid,)).fetchone()
    df = pd.read_sql("SELECT start_ts, sum FROM statistics WHERE metadata_id=? AND start_ts>=? ORDER BY start_ts",
                     con, params=(mid, time.time() - DAGEN * 86400))
    idx = pd.DatetimeIndex(pd.to_datetime(df.start_ts, unit="s", utc=True)).tz_convert("Europe/Amsterdam")
    s = pd.Series(df["sum"].values, index=idx).diff().clip(lower=0)
    return s / 1000 if unit == "Wh" else s

def som(ids):
    out = None
    for sid in ids:
        out = reeks(sid) if out is None else out.add(reeks(sid), fill_value=0)
    return out

uur = pd.DataFrame({"afname": som(IMPORT_IDS), "teruglevering": som(EXPORT_IDS)}).dropna().asfreq("h", fill_value=0)
uur["pv_uit"] = uur.teruglevering < GEEN_TERUG
IMP, EXP, UIT = uur.afname.values, uur.teruglevering.values, uur.pv_uit.values
print(f"{uur.index.min():%d-%m-%Y} t/m {uur.index.max():%d-%m-%Y}, {len(uur)} uren")

In [ ]:
def geleverd(imp, exp, uit, cap, kw, eta=RENDEMENT):
    eff = eta ** 0.5
    soc = tot = 0.0
    for i, e, u in zip(imp, exp, uit):
        if cap > soc:
            soc += min(e, kw, (cap - soc) / eff) * eff
        if u:
            ont = min(i, kw, soc * eff)
            soc -= ont / eff
            tot += ont
    return tot

In [ ]:
# Tarieven (vast contract) en aanschafprijzen
INKOOP = 0.26465                  # EUR/kWh
TERUG_NETTO = 0.141 - 0.13099     # EUR/kWh: vergoeding minus terugleverkosten
SB_W = 1.0                        # stand-by per module (W)
rijen = []
for naam, cap, kw, n, prijs in [("1 module (2,7 kWh, 0,8 kW)", 2.47, 0.8, 1, 1000),
                                ("2 modules (5,4 kWh, 1,6 kW)", 4.94, 1.6, 2, 2000)]:
    for eta in (0.70, 0.75, 0.80):
        lev = geleverd(IMP, EXP, UIT, cap, kw, eta) * 365 * 24 / len(IMP)
        netto = lev * INKOOP - lev / eta * TERUG_NETTO - n * SB_W * 8.76 * INKOOP
        rijen.append(dict(config=naam, rendement=eta, geleverd_kwh_jaar=round(lev), geladen_kwh_jaar=round(lev / eta),
                          netto_eur_jaar=round(netto), terugverdientijd_jaar=round(prijs / netto, 1)))
res = pd.DataFrame(rijen); res.to_csv("homewizard_scan.csv", sep=";", decimal=",", index=False)
print(res.to_string(index=False))